# Phase 4 Final — Phase 2 top-80 + Vietnamese Legal BGE + protected rank blender

Attach the competition data, Phase 2 Harrier bundle, Phase 3 reranker delta bundle, and the saved output of the completed Phase 2 private notebook. Select RTX Pro 6000 and set Internet to Off.

The notebook reuses every expensive Phase 2 retrieval/Jina/Vietnamese-reranker artifact. It runs only the small Vietnamese legal BGE reranker over the existing top-80 candidates, then trains a cross-fitted CPU rank blender with protected promotion from the proven Phase 2 top five. Previous submissions are never modified.


In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from pathlib import Path
EXPERIMENT_ID = 'phase4-phase2-legal-bge-protected-blender'
DATASET_DIR = Path('/kaggle/input/datasets/tonioz/uit-dsc-task1')
TEST_FILENAME = 'private-official.json'
if TEST_FILENAME not in {'private-official.json', 'public-official.json'}:
    raise ValueError(TEST_FILENAME)
TEST_LABEL = 'private' if TEST_FILENAME == 'private-official.json' else 'public'
PHASE2_BUNDLE = Path('/kaggle/input/datasets/boinhbo/legalir-phase2-harrier-bundle/legalir-phase2-harrier-bundle')
PHASE3_DELTA = Path('/kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta')
PHASE2_ARTIFACTS_OVERRIDE = None  # Set Path(...) only if auto-discovery finds multiple complete copies.
WORK_DIR = Path(f'/kaggle/working/legalir-phase4-{TEST_LABEL}-final')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase4-runtime')


In [ ]:
import json
import shutil
import subprocess
import sys
import time

def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')

phase2_manifest = json.loads((PHASE2_BUNDLE / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
delta_manifest = json.loads((PHASE3_DELTA / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
if phase2_manifest.get('experiment_id') != 'phase2-vietlegal-harrier-0.6b':
    raise RuntimeError('Wrong Phase 2 bundle')
if delta_manifest.get('experiment_id') != 'phase3-rerankers-harrier-retrieval':
    raise RuntimeError('Wrong Phase 3 delta bundle')
if 'legal_reranker' not in {row['name'] for row in delta_manifest['models']}:
    raise RuntimeError('Phase 3 delta lacks legal_reranker')
for manifest, bundle in ((phase2_manifest, PHASE2_BUNDLE), (delta_manifest, PHASE3_DELTA)):
    for record in manifest['files']:
        path = bundle / record['path']
        if not path.is_file() or path.stat().st_size != record['bytes']:
            raise RuntimeError(f'Missing or truncated bundle file: {path}')

WORK_DIR.mkdir(parents=True, exist_ok=True)
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', PHASE3_DELTA / 'wheels', '-r', PHASE3_DELTA / 'requirements-offline.txt')
project_wheels = sorted((PHASE3_DELTA / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1:
    raise RuntimeError(f'Expected one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
probe = "import sklearn, torch, transformers; assert torch.cuda.is_available(); print('sklearn', sklearn.__version__); print('GPU', torch.cuda.get_device_name(0)); print('transformers', transformers.__version__)"
run(sys.executable, '-c', probe, env=runtime_env)
print('Runtime project commit:', delta_manifest['project_commit'])


In [ ]:
import hashlib
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir():
    raise FileNotFoundError(f'Missing corpus: {contexts_source}')
for filename in ('train.json', TEST_FILENAME):
    if not (DATASET_DIR / filename).is_file():
        raise FileNotFoundError(DATASET_DIR / filename)

def ensure_link(destination, source, is_directory=False):
    if destination.is_symlink():
        if destination.resolve() == source.resolve():
            return
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(f'Refusing to overwrite: {destination}')
    destination.symlink_to(source, target_is_directory=is_directory)

ensure_link(WORK_DIR / 'selected-contexts', contexts_source, True)
for filename in ('train.json', TEST_FILENAME):
    ensure_link(WORK_DIR / filename, DATASET_DIR / filename)
test_bytes = (DATASET_DIR / TEST_FILENAME).read_bytes()
test_sha256 = hashlib.sha256(test_bytes).hexdigest()
test_question_count = len(json.loads(test_bytes))
print('Corpus documents:', sum(1 for _ in contexts_source.glob('context_*.json')))
print('Test questions:', test_question_count, 'sha256:', test_sha256)

required_phase2 = {
    'prepare_manifest.json', 'corpus.jsonl', 'chunks_short.jsonl', 'chunks_long.jsonl',
    'lexical_short.pkl', 'train_questions.jsonl', 'public_questions.jsonl',
    'first_stage_weights.json', 'final_weights.json', 'retrieval_train.json',
    'retrieval_public.json', 'rerank_train_0.json', 'rerank_public.json',
}
def complete_phase2(path):
    return path.is_dir() and all((path / name).is_file() for name in required_phase2) and all(
        all((path / 'dense' / model / name).is_file() for name in ('vectors.npy', 'index.faiss', 'chunks.json'))
        for model in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron')
    ) and all(
        all((path / 'question_memory' / model / name).is_file() for name in ('vectors.npy', 'questions.json'))
        for model in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron')
    )

if PHASE2_ARTIFACTS_OVERRIDE is not None:
    PHASE2_ARTIFACTS = Path(PHASE2_ARTIFACTS_OVERRIDE)
    if not complete_phase2(PHASE2_ARTIFACTS):
        raise RuntimeError(f'Incomplete Phase 2 artifacts: {PHASE2_ARTIFACTS}')
else:
    candidates = [path for path in Path('/kaggle/input').rglob('artifacts_phase2_harrier') if complete_phase2(path)]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one complete Phase 2 artifact directory, found {candidates}. Set PHASE2_ARTIFACTS_OVERRIDE.')
    PHASE2_ARTIFACTS = candidates[0]

state_path = PHASE2_ARTIFACTS.parent / 'inference_input_state.json'
if not state_path.is_file():
    raise RuntimeError(f'Phase 2 output lacks inference input state: {state_path}')
state = json.loads(state_path.read_text(encoding='utf-8'))
if state.get('filename') != TEST_FILENAME or state.get('sha256') != test_sha256 or state.get('questions') != test_question_count:
    raise RuntimeError(f'Phase 2 private cache fingerprint mismatch: {state}')
print('Using complete matching Phase 2 artifacts:', PHASE2_ARTIFACTS)


In [ ]:
import yaml
config = yaml.safe_load((PHASE2_BUNDLE / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
config['paths']['public_file'] = TEST_FILENAME
config['paths']['artifacts_dir'] = 'artifacts_phase4'
for name in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron', 'jina', 'vietnamese_reranker'):
    config['models'][name]['local_path'] = str(PHASE2_BUNDLE / 'models' / name)
    config['models'][name]['local_files_only'] = True
phase3_config = yaml.safe_load((PHASE3_DELTA / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
config['models']['legal_reranker'] = phase3_config['models']['legal_reranker']
config['models']['legal_reranker']['local_path'] = str(PHASE3_DELTA / 'models' / 'legal_reranker')
config['models']['legal_reranker']['local_files_only'] = True
config['reranking']['rerank_top_k'] = 80
config['reranking']['pairwise_max_length'] = 512
config['reranking']['pairwise_evidence_tokens'] = 440
config['reranking']['pairwise_batch_size'] = 64
config['reranking']['query_batch_size'] = 16
config['reranking']['checkpoint_every_questions'] = 32
config['validation']['reranker_tuning_folds'] = [0]

artifacts = WORK_DIR / config['paths']['artifacts_dir']
artifacts.mkdir(parents=True, exist_ok=True)
for source in PHASE2_ARTIFACTS.iterdir():
    if source.name in {'model_manifest.json', 'final_weights.json'}:
        continue
    if source.name.startswith('rerank_train_0_legal_reranker') or source.name.startswith('rerank_public_legal_reranker'):
        continue
    destination = artifacts / source.name
    if not destination.exists() and not destination.is_symlink():
        destination.symlink_to(source, target_is_directory=source.is_dir())
shutil.copy2(PHASE2_ARTIFACTS / 'final_weights.json', artifacts / 'phase2_final_weights.json')
config_path = WORK_DIR / 'kaggle_rtx_pro_6000_phase4.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))


In [ ]:
# Only the newly added model needs a preflight; Phase 2 models already produced the attached caches.
preflight = WORK_DIR / 'phase4_legal_preflight.py'
preflight.write_text("import sys\nfrom pathlib import Path\nimport yaml\nfrom legalir.rerank import PairwiseReranker\nconfig = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))\nengine = PairwiseReranker(config, 'legal_reranker')\norder = engine.rank('điều kiện cấp giấy phép', ['văn bản có quy định cấp giấy phép', 'văn bản không liên quan'])\nassert sorted(order) == [0, 1]\nengine.close()\nprint('Phase 4 legal reranker preflight passed')\n", encoding='utf-8')
run(sys.executable, preflight, config_path, cwd=WORK_DIR, env=runtime_env)

base = [sys.executable, '-m', 'legalir']
def legalir(*args):
    run(*base, *args, cwd=WORK_DIR, env=runtime_env)

# Audit loads/counts all deployed checkpoints but does not recompute retrieval or Phase 2 reranking.
legalir('audit', '--config', config_path)
legalir('rerank', '--config', config_path, '--split', 'train', '--fold', '0', '--engine', 'legal_reranker', '--resume')
legalir('rerank', '--config', config_path, '--split', 'public', '--engine', 'legal_reranker', '--resume')


In [ ]:
blender_script = WORK_DIR / 'phase4_blend.py'
blender_script.write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport pickle\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport yaml\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom legalir.fusion import rrf\nfrom legalir.storage import read_json, read_jsonl, write_json\nfrom legalir.text import normalize_question\nfrom legalir.validation import grouped_folds, score_candidates, score_predictions, validate_submission_shape\n\n\nRERANKERS = ("jina", "vietnamese_reranker", "legal_reranker")\nRETRIEVAL_CHANNELS = (\n    "bm25",\n    "accent_char",\n    "vietlegal_harrier",\n    "vietnamese_embedding",\n    "nemotron",\n    "query_memory",\n    "query_exact",\n)\n\n\ndef fuse_cached(\n    retrievals: dict[str, dict[str, Any]],\n    weights: dict[str, float],\n    rrf_k: int,\n    limit: int,\n) -> dict[str, dict[str, list[str]]]:\n    return {\n        qid: {"candidates": rrf(retrieval["channels"], weights, rrf_k, limit)}\n        for qid, retrieval in retrievals.items()\n    }\n\n\ndef engine_ranks(artifacts: Path, split: str, engine: str, fold: int | None = None) -> dict[str, list[str]]:\n    suffix = f"_{fold}" if fold is not None else ""\n    separate = artifacts / f"rerank_{split}{suffix}_{engine}.json"\n    if separate.is_file():\n        return read_json(separate)[engine]\n    combined = artifacts / f"rerank_{split}{suffix}.json"\n    payload = read_json(combined)\n    if engine not in payload:\n        raise RuntimeError(f"{engine} is missing from {combined}")\n    return payload[engine]\n\n\ndef inner_fold(question: str) -> int:\n    # A salt different from grouped_folds is essential: every selected question\n    # is already in outer fold zero under the main fold hash.\n    key = "phase4-inner-v1:" + normalize_question(question)\n    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:12], 16) % 5\n\n\ndef reciprocal_rank(rank: int, k: int = 20) -> float:\n    return 1.0 / (k + rank)\n\n\ndef rank_features(rank: int, missing_rank: int) -> list[float]:\n    clipped = min(rank, missing_rank)\n    denominator = max(1, missing_rank - 1)\n    return [\n        reciprocal_rank(clipped),\n        1.0 / clipped,\n        1.0 - min(clipped - 1, denominator) / denominator,\n        float(clipped <= 5),\n        float(clipped <= 10),\n        float(clipped <= 20),\n        float(clipped <= 50),\n        float(clipped < missing_rank),\n    ]\n\n\ndef feature_vector(document: str, orders: dict[str, dict[str, int]]) -> list[float]:\n    features: list[float] = []\n    core_ranks: list[int] = []\n    for name in ("first_stage", *RERANKERS):\n        rank = orders[name].get(document, 81)\n        core_ranks.append(rank)\n        features.extend(rank_features(rank, 81))\n    retrieval_ranks: list[int] = []\n    for name in RETRIEVAL_CHANNELS:\n        rank = orders[name].get(document, 151)\n        retrieval_ranks.append(rank)\n        features.extend(rank_features(rank, 151))\n\n    all_ranks = core_ranks + retrieval_ranks\n    features.extend(\n        [\n            float(sum(rank <= 5 for rank in core_ranks)),\n            float(sum(rank <= 10 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in retrieval_ranks)),\n            float(sum(rank < 151 for rank in retrieval_ranks)),\n            float(min(all_ranks)),\n            float(max(core_ranks)),\n            float(np.mean(core_ranks)),\n            float(np.std(core_ranks)),\n        ]\n    )\n    phase2_score = (\n        0.3 * reciprocal_rank(core_ranks[0])\n        + 0.5 * reciprocal_rank(core_ranks[1])\n        + 0.5 * reciprocal_rank(core_ranks[2])\n    )\n    legal_residual = reciprocal_rank(core_ranks[3]) - reciprocal_rank(core_ranks[0])\n    features.extend([phase2_score, legal_residual])\n    return features\n\n\ndef make_rows(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    retrievals: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    labelled: bool,\n) -> tuple[np.ndarray, np.ndarray, list[tuple[str, str]], np.ndarray]:\n    rows: list[list[float]] = []\n    labels: list[int] = []\n    metadata: list[tuple[str, str]] = []\n    groups: list[int] = []\n    for question in questions:\n        qid = question["qid"]\n        candidates = fused[qid]["candidates"][:80]\n        orders = {"first_stage": {doc: rank for rank, doc in enumerate(candidates, 1)}}\n        for name, values in rerankings.items():\n            orders[name] = {doc: rank for rank, doc in enumerate(values[qid], 1)}\n        for name in RETRIEVAL_CHANNELS:\n            orders[name] = {doc: rank for rank, doc in enumerate(retrievals[qid]["channels"][name], 1)}\n        truth = set(question.get("answers", []))\n        group = inner_fold(question["question"])\n        for document in candidates:\n            rows.append(feature_vector(document, orders))\n            labels.append(int(document in truth) if labelled else 0)\n            metadata.append((qid, document))\n            groups.append(group)\n    return (\n        np.asarray(rows, dtype=np.float32),\n        np.asarray(labels, dtype=np.int8),\n        metadata,\n        np.asarray(groups, dtype=np.int8),\n    )\n\n\ndef standardize_per_query(values: np.ndarray, metadata: list[tuple[str, str]]) -> np.ndarray:\n    result = np.zeros(len(values), dtype=np.float64)\n    by_qid: defaultdict[str, list[int]] = defaultdict(list)\n    for index, (qid, _) in enumerate(metadata):\n        by_qid[qid].append(index)\n    for indices in by_qid.values():\n        scores = values[indices]\n        scale = scores.std()\n        result[indices] = (scores - scores.mean()) / (scale if scale > 1e-9 else 1.0)\n    return result\n\n\ndef scores_by_question(values: np.ndarray, metadata: list[tuple[str, str]]) -> dict[str, dict[str, float]]:\n    output: defaultdict[str, dict[str, float]] = defaultdict(dict)\n    for score, (qid, document) in zip(values, metadata, strict=True):\n        output[qid][document] = float(score)\n    return dict(output)\n\n\ndef protected_predictions(\n    blended_scores: np.ndarray,\n    metadata: list[tuple[str, str]],\n    baseline: dict[str, list[str]],\n    margin: float,\n) -> tuple[dict[str, list[str]], dict[str, int]]:\n    per_query = scores_by_question(blended_scores, metadata)\n    output: dict[str, list[str]] = {}\n    promoted_questions = 0\n    promotions = 0\n    for qid, scores in per_query.items():\n        selected = list(baseline[qid])\n        outsiders = [doc for doc, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0])) if doc not in selected]\n        changed = False\n        for outsider in outsiders:\n            weakest = min(selected, key=lambda doc: (scores[doc], doc))\n            if scores[outsider] < scores[weakest] + margin:\n                break\n            selected[selected.index(weakest)] = outsider\n            promotions += 1\n            changed = True\n        if changed:\n            promoted_questions += 1\n        # Ordering is irrelevant to Recall, but sorting makes the output deterministic.\n        output[qid] = sorted(selected, key=lambda doc: (-scores[doc], doc))\n    return output, {"promoted_questions": promoted_questions, "promotions": promotions}\n\n\ndef new_model(c_value: float):\n    return make_pipeline(\n        StandardScaler(),\n        LogisticRegression(\n            C=c_value,\n            class_weight="balanced",\n            max_iter=600,\n            solver="liblinear",\n            random_state=2026,\n        ),\n    )\n\n\ndef baseline_predictions(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    final_weights: dict[str, Any],\n) -> dict[str, list[str]]:\n    return {\n        question["qid"]: rrf(\n            {\n                "first_stage": fused[question["qid"]]["candidates"],\n                "jina": rerankings["jina"][question["qid"]],\n                "vietnamese_reranker": rerankings["vietnamese_reranker"][question["qid"]],\n            },\n            final_weights["weights"],\n            final_weights["rrf_k"],\n            5,\n        )\n        for question in questions\n    }\n\n\ndef main() -> None:\n    work = Path(sys.argv[1])\n    config = yaml.safe_load(Path(sys.argv[2]).read_text(encoding="utf-8"))\n    artifacts = Path(config["paths"]["artifacts_dir"])\n    if not artifacts.is_absolute():\n        artifacts = work / artifacts\n\n    first_stage = read_json(artifacts / "first_stage_weights.json")\n    phase2_final = read_json(artifacts / "phase2_final_weights.json")\n    retrieval_train = read_json(artifacts / "retrieval_train.json")\n    retrieval_public = read_json(artifacts / "retrieval_public.json")\n    fused_limit = int(config["retrieval"]["fused_top_k"])\n    fused_train = fuse_cached(retrieval_train, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    fused_public = fuse_cached(retrieval_public, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    train_questions = list(read_jsonl(artifacts / "train_questions.jsonl"))\n    public_questions = list(read_jsonl(artifacts / "public_questions.jsonl"))\n    outer_folds = grouped_folds(train_questions, config["validation"]["folds"])\n    fold_questions = [question for question in train_questions if outer_folds[question["qid"]] == 0]\n\n    train_rerankings = {name: engine_ranks(artifacts, "train", name, 0) for name in RERANKERS}\n    public_rerankings = {name: engine_ranks(artifacts, "public", name) for name in RERANKERS}\n    for questions, rerankings, split in (\n        (fold_questions, train_rerankings, "train"),\n        (public_questions, public_rerankings, "public"),\n    ):\n        expected = {question["qid"] for question in questions}\n        for name, values in rerankings.items():\n            missing = expected.difference(values)\n            wrong_depth = [qid for qid in expected.intersection(values) if len(values[qid]) != 80]\n            if missing or wrong_depth:\n                raise RuntimeError(f"{split}/{name}: missing={len(missing)}, non_top80={len(wrong_depth)}")\n\n    baseline = baseline_predictions(fold_questions, fused_train, train_rerankings, phase2_final)\n    baseline_metrics = score_predictions(baseline, fold_questions)\n    if any(not set(baseline[q["qid"]]).issubset(fused_train[q["qid"]]["candidates"][:80]) for q in fold_questions):\n        raise RuntimeError("Phase 2 baseline selected a document outside top 80")\n\n    x_train, y_train, train_metadata, inner_groups = make_rows(\n        fold_questions, fused_train, retrieval_train, train_rerankings, True\n    )\n    if y_train.sum() == 0 or set(inner_groups) != set(range(5)):\n        raise RuntimeError(\n            f"Invalid training sample: positives={int(y_train.sum())}, inner_folds={sorted(set(inner_groups))}"\n        )\n    anchor_train = standardize_per_query(x_train[:, -2].astype(np.float64), train_metadata)\n    baseline_per_inner = {\n        str(fold): score_predictions(\n            baseline,\n            [question for question in fold_questions if inner_fold(question["question"]) == fold],\n        )\n        for fold in range(5)\n    }\n\n    trials: list[dict[str, Any]] = []\n    for c_value in (0.03, 0.1, 0.3, 1.0, 3.0):\n        oof = np.zeros(len(y_train), dtype=np.float64)\n        for heldout in range(5):\n            train_mask = inner_groups != heldout\n            valid_mask = inner_groups == heldout\n            model = new_model(c_value)\n            model.fit(x_train[train_mask], y_train[train_mask])\n            oof[valid_mask] = model.decision_function(x_train[valid_mask])\n        learned = standardize_per_query(oof, train_metadata)\n        for alpha in (0.35, 0.5, 0.65, 0.8, 1.0):\n            blended = alpha * learned + (1.0 - alpha) * anchor_train\n            for margin in (0.0, 0.25, 0.5, 0.75, 1.0):\n                prediction, promotion = protected_predictions(blended, train_metadata, baseline, margin)\n                per_inner = {\n                    str(fold): score_predictions(\n                        prediction,\n                        [question for question in fold_questions if inner_fold(question["question"]) == fold],\n                    )\n                    for fold in range(5)\n                }\n                deltas = [per_inner[str(fold)]["recall"] - baseline_per_inner[str(fold)]["recall"] for fold in range(5)]\n                trials.append(\n                    {\n                        "C": c_value,\n                        "alpha": alpha,\n                        "promotion_margin": margin,\n                        "metrics": score_predictions(prediction, fold_questions),\n                        "per_inner": per_inner,\n                        "nonnegative_inner_folds": sum(delta >= -1e-12 for delta in deltas),\n                        "worst_inner_recall_delta": min(deltas),\n                        **promotion,\n                    }\n                )\n\n    robust = [\n        row\n        for row in trials\n        if row["nonnegative_inner_folds"] >= 4 and row["worst_inner_recall_delta"] >= -0.005\n    ]\n    pool = robust or trials\n    best = max(\n        pool,\n        key=lambda row: (\n            row["metrics"]["recall"],\n            row["nonnegative_inner_folds"],\n            row["worst_inner_recall_delta"],\n            row["metrics"]["precision"],\n            -row["promotions"],\n            -row["C"],\n        ),\n    )\n\n    final_model = new_model(best["C"])\n    final_model.fit(x_train, y_train)\n    x_public, _, public_metadata, _ = make_rows(\n        public_questions, fused_public, retrieval_public, public_rerankings, False\n    )\n    learned_public = standardize_per_query(final_model.decision_function(x_public), public_metadata)\n    anchor_public = standardize_per_query(x_public[:, -2].astype(np.float64), public_metadata)\n    baseline_public = baseline_predictions(public_questions, fused_public, public_rerankings, phase2_final)\n    blended_public = best["alpha"] * learned_public + (1.0 - best["alpha"]) * anchor_public\n    final_prediction, private_promotion = protected_predictions(\n        blended_public, public_metadata, baseline_public, best["promotion_margin"]\n    )\n\n    corpus_ids = {row["doc_id"] for row in read_jsonl(artifacts / "corpus.jsonl")}\n    submission = {qid: {"answer": documents} for qid, documents in final_prediction.items()}\n    validate_submission_shape(submission, public_questions, corpus_ids)\n    submission_path = work / "submission_phase4_private_final.json"\n    write_json(submission_path, submission)\n    with (work / "phase4_blender.pkl").open("wb") as handle:\n        pickle.dump(final_model, handle)\n\n    report = {\n        "experiment_id": "phase4-phase2-legal-bge-protected-blender",\n        "training_questions": len(fold_questions),\n        "training_pairs": len(y_train),\n        "positive_pairs": int(y_train.sum()),\n        "feature_count": int(x_train.shape[1]),\n        "inner_fold_counts": {\n            str(fold): int(sum(inner_fold(question["question"]) == fold for question in fold_questions))\n            for fold in range(5)\n        },\n        "candidate_top_k": 80,\n        "candidate_recall_fold0": score_candidates(\n            {question["qid"]: fused_train[question["qid"]]["candidates"][:80] for question in fold_questions},\n            fold_questions,\n        )["candidate_recall"],\n        "phase2_baseline_fold0": baseline_metrics,\n        "phase2_baseline_per_inner": baseline_per_inner,\n        "selected_cross_fitted": best,\n        "recall_delta": best["metrics"]["recall"] - baseline_metrics["recall"],\n        "robust_trials": len(robust),\n        "tested_configurations": len(trials),\n        "private_promotions": private_promotion,\n        "submission": str(submission_path),\n    }\n    write_json(work / "phase4_report.json", report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()', encoding='utf-8')
run(sys.executable, blender_script, WORK_DIR, config_path, cwd=WORK_DIR, env=runtime_env)
submission_json = WORK_DIR / 'submission_phase4_private_final.json'
submission_zip = WORK_DIR / 'submission_phase4_private_final.zip'
run('zip', '-j', submission_zip, submission_json)
print('PHASE 4 FINAL:', submission_zip)


In [ ]:
report = json.loads((WORK_DIR / 'phase4_report.json').read_text(encoding='utf-8'))
model_manifest = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
submission = json.loads((WORK_DIR / 'submission_phase4_private_final.json').read_text(encoding='utf-8'))
phase2_submission_path = PHASE2_ARTIFACTS.parent / 'submission_phase2_private_harrier.json'
comparison = None
if phase2_submission_path.is_file():
    phase2_submission = json.loads(phase2_submission_path.read_text(encoding='utf-8'))
    shared = set(submission).intersection(phase2_submission)
    comparison = {
        'questions_compared': len(shared),
        'changed_answer_sets': sum(set(submission[q]['answer']) != set(phase2_submission[q]['answer']) for q in shared),
        'changed_top1': sum(submission[q]['answer'][0] != phase2_submission[q]['answer'][0] for q in shared),
    }
report.update({
    'test_file': TEST_FILENAME,
    'test_questions': test_question_count,
    'test_sha256': test_sha256,
    'project_commit': delta_manifest['project_commit'],
    'models': model_manifest['models'],
    'total_parameters': model_manifest['total_parameters'],
    'private_comparison_to_phase2': comparison,
    'submission_zip': str(WORK_DIR / 'submission_phase4_private_final.zip'),
})
(WORK_DIR / 'phase4_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
